In [1]:
import random, os
import numpy as np
import torch
os.environ["CUDA_VISIBLE_DEVICES"]="3"

from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import pandas as pd
import re

device1 = 'cuda:0'
device2 = 'cuda:1'
data_dir = '/raid/deallab/SF_RAG_Data/ASQA'
# data_dir = '../data'

/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed_value):
    # Set seed for reproducibility.
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED']=str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.backends.cudnn.deterministic=True    
    torch.backends.cudnn.benchmark=True
    torch.cuda.manual_seed_all(seed_value)

In [3]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load embeddings (evidence_eval)
embedd_test_path = f'{data_dir}/test/embedd_test_eval2.npy'
evidence_embeddings_eval = np.load(embedd_test_path)
print(evidence_embeddings_eval.shape)
evidence_embeddings_eval = torch.from_numpy(evidence_embeddings_eval).to(device1)

#load embeddings (evidence_title)
embedd_test_path = f'{data_dir}/test/embedd_test_eval_title.npy'
evidence_embeddings_title = np.load(embedd_test_path)
print(evidence_embeddings_title.shape)
evidence_embeddings_title = torch.from_numpy(evidence_embeddings_title).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)
print(len(evidence_df))

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test_eval.csv'
evidence_eval_df = pd.read_csv(evidence_test_path)
print(len(evidence_eval_df))

#load title evidence
evidence_test_path = f'{data_dir}/test/evidence_test_eval_title.csv'
evidence_eval_title_df = pd.read_csv(evidence_test_path)
print(len(evidence_eval_title_df))

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(21586, 4096)
(21801, 4096)
(1897, 4096)
21586
21801
1897


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,aee000f3-d2b0-4de5-8206-a96e9c203207,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,b80ea7a2-5084-422f-94d1-e67e7e29819b,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,a9c1319b-ed69-4b1b-8486-05ad3d444e22,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,b07f0006-0628-49cf-b5b4-c0c7e88d3190,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,a9d31e99-6402-4a41-a4af-52de1aebeb16,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [4]:
evidence_eval_title_df

,sample_id,title,text
0,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
1,-7013890438520559398,List of FIFA World Cup records and statistics,Document: List of FIFA World Cup records and s...
2,-7013890438520559398,List of footballers with more than 50 internat...,Document: List of footballers with more than 5...
3,-7013890438520559398,List of women's footballers with 100 or more i...,Document: List of women's footballers with 100...
4,-7013890438520559398,List of footballers with 500 or more goals,Document: List of footballers with 500 or more...
...,...,...,...
1892,8651936974639934250,Singin' in the Rain (musical),Document: Singin' in the Rain (musical)\n\nSin...
1893,8651936974639934250,Singin' in the Rain (song),Document: Singin' in the Rain (song)\n\n\n\n\n...
1894,8651936974639934250,Jimmy Thompson (actor),# Jimmy Thompson (actor) #\nJames Edward Thomp...
1895,8651936974639934250,Singin' in the Rain,Document: Singin' in the Rain\n\nSingin' in th...


In [5]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.19s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [6]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
# cache_dir= '/raid/deallab/.cache')
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map= 'auto',
    # cache_dir= '/raid/deallab/.cache'
)
model_gen.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.37it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): Ll

In [10]:
# from langchain_text_splitters import TokenTextSplitter

# text_splitter = TokenTextSplitter(
#     chunk_size=500,  # 청크 크기를 10으로 설정합니다.
#     chunk_overlap=50,  # 청크 간 중복을 0으로 설정합니다.
# )
# # combined_text = " ".join(evidence_text_list)
# # texts = text_splitter.split_text(combined_text)
# split_texts = [text_splitter.split_text(text)[0] for text in evidence_text_list]
# print(split_texts[0])

In [11]:
# from langchain.retrievers import BM25Retriever, EnsembleRetriever
# from langchain.vectorstores import FAISS

# # bm25 retriever와 faiss retriever를 초기화합니다.
# bm25_retriever = BM25Retriever.from_texts(
#     evidence_text_list,
# )
# bm25_retriever.k = 10  # BM25Retriever의 검색 결과 개수를 1로 설정합니다.

# embedding = model
# faiss_vectorstore = FAISS.from_texts(
#     evidence_text,
#     embedding,
# )
# faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

# # 앙상블 retriever를 초기화합니다.
# ensemble_retriever = EnsembleRetriever(
#     retrievers=[bm25_retriever, faiss_retriever],
#     weights=[0.7, 0.3],
# )

In [12]:
# from langchain_community.document_transformers import LongContextReorder

# def bm25_retrieve(query):
#     bm25_result = bm25_retriever.invoke(query)
#     bm25_docs=list()

#     print("[BM25 Retriever]")
#     for doc in bm25_result:
#         # print(f"Content: {doc.page_content}")
#         # print()
#         bm25_docs.append(doc.page_content)
#     reordering = LongContextReorder()
#     bm25_docs = reordering.transform_documents(bm25_docs)
#     return bm25_docs

In [13]:
# res=bm25_retrieve("Who has the highest goals in world football?")
# res

In [7]:
evidence_eval_path = f'{data_dir}/test/evidence_test_eval.csv'
eval_df = pd.read_csv(evidence_eval_path)
eval_df.head(20)

,sample_id,title,text
0,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
1,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
2,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
3,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
4,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
5,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
6,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
7,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
8,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
9,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...


In [8]:
def match_sample_id(query_id, doc_id):
    query_sample_id=qa_df.loc[query_id,'sample_id']
    doc_sample_id=eval_df.loc[doc_id,'sample_id']
    if query_sample_id==doc_sample_id:
        return 1
    else:
        return 0
    

In [9]:
evidence_eval_title_df

,sample_id,title,text
0,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
1,-7013890438520559398,List of FIFA World Cup records and statistics,Document: List of FIFA World Cup records and s...
2,-7013890438520559398,List of footballers with more than 50 internat...,Document: List of footballers with more than 5...
3,-7013890438520559398,List of women's footballers with 100 or more i...,Document: List of women's footballers with 100...
4,-7013890438520559398,List of footballers with 500 or more goals,Document: List of footballers with 500 or more...
...,...,...,...
1892,8651936974639934250,Singin' in the Rain (musical),Document: Singin' in the Rain (musical)\n\nSin...
1893,8651936974639934250,Singin' in the Rain (song),Document: Singin' in the Rain (song)\n\n\n\n\n...
1894,8651936974639934250,Jimmy Thompson (actor),# Jimmy Thompson (actor) #\nJames Edward Thomp...
1895,8651936974639934250,Singin' in the Rain,Document: Singin' in the Rain\n\nSingin' in th...


In [10]:
def match_sample_id2(query_id, title):
    query_sample_id=qa_df.loc[query_id,'sample_id']
    doc_sample_id=evidence_eval_title_df.loc[evidence_eval_title_df['title']==title,'sample_id'].item()
    if query_sample_id==doc_sample_id:
        return 1
    else:
        return 0

In [11]:
# retrive docs from the document embeddings
def retrieve_documents(query,num=10):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:num].cpu().detach().numpy()
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
    idx=[idx for idx in top_results if idx < len(evidence_df)]
    return res, idx

In [12]:
evidence_eval_df

,sample_id,title,text
0,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
1,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
2,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
3,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
4,-7013890438520559398,International Federation of Football History &...,Document: International Federation of Football...
...,...,...,...
21796,4198094437803531947,Sign of the Times (Harry Styles song),Document: Sign of the Times (Harry Styles song...
21797,4198094437803531947,Sign of the Times (Harry Styles song),Document: Sign of the Times (Harry Styles song...
21798,4198094437803531947,Sign of the Times (Harry Styles song),Document: Sign of the Times (Harry Styles song...
21799,4198094437803531947,Sign of the Times (Harry Styles song),Document: Sign of the Times (Harry Styles song...


In [13]:
# retrive docs from the document embeddings
def retrieve_documents_eval(query,num=10):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings_eval)

    top_results = similarities.argsort(descending=True)[:num].cpu().detach().numpy()
    res=[evidence_eval_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_eval_df)]
    idx=[idx for idx in top_results if idx < len(evidence_eval_df)]
    return res, idx

In [14]:
# retrive docs from the document embeddings
def retrieve_documents2(query, embed, evidence, num=10):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, embed)

    top_results = similarities.argsort(descending=True)[:num].cpu().detach().numpy()
    res=[evidence.loc[idx, 'text'] for idx in top_results if idx < len(evidence)]
    titles=[evidence.loc[idx, 'title'] for idx in top_results if idx < len(evidence)]
    # idx=[idx for idx in top_results if idx < len(evidence)]
    return res, titles

In [15]:
# retrive docs from the document embeddings
def retrieve_documents3(query, embed, evidence, num=10):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, embed)

    top_results = similarities.argsort(descending=True)[:num].cpu().detach().numpy()
    res=[evidence.loc[idx, 'text'] for idx in top_results if idx < len(evidence)]
    # titles=[evidence.loc[idx, 'title'] for idx in top_results if idx < len(evidence)]
    idx=[idx for idx in top_results if idx < len(evidence)]
    return res, idx

In [16]:
# retrive docs from the document embeddings
def retrieve_titles(query,num=10):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings_title)

    top_results = similarities.argsort(descending=True)[:num].cpu().detach().numpy()
    res=[evidence_eval_title_df.loc[idx, 'title'] for idx in top_results if idx < len(evidence_eval_title_df)]
    idx=[idx for idx in top_results if idx < len(evidence_eval_title_df)]
    return res, idx

In [17]:
def find_titles(query):
    titles,title_ids=retrieve_titles(query, 10)
    return titles

In [18]:
def find_docs(titles):
    docs=[]
    for title in titles:
        # print(title)
        # print(evidence_eval_df.loc[evidence_eval_df['title']==title,'text'].to_list())
        docs.extend(evidence_eval_df.loc[evidence_eval_df['title']==title,'text'])
        # print()
    return docs

In [19]:
def retrieve_inside(query, titles, num=10):
    docs=[]
    text_docs=pd.DataFrame(columns=['text','title'])
    final_docs=[]
    for title in titles:
        text_docs=pd.concat([text_docs, evidence_eval_df.loc[evidence_eval_df['title']==title,['text','title']]],ignore_index=True)
        docs.extend(evidence_embeddings_eval[evidence_eval_df['title']==title])
        # print(title_retrieved_docs)
        # print()
    docs_tensor = torch.stack(docs)
    title_retrieved_docs,t=retrieve_documents2(query, docs_tensor, text_docs, num)
    final_docs.extend(title_retrieved_docs)
    return final_docs,t

In [20]:
query="Who has the highet goals in world football?"
titles,title_ids=retrieve_titles(query, 11)
for t in titles:
    print(t)
title_docs=find_docs(titles)
final_docs,d=retrieve_inside(query, titles, 10)
d

/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


List of men's footballers with 50 or more international goals
List of footballers with more than 50 international goals
List of footballers with 500 or more goals
FIFA World Cup top goalscorers
List of top international men's association football goal scorers by ...
List of women's footballers with 100 or more international goals ...
List of FIFA World Cup records and statistics
List of UEFA Champions League top scorers
List of FIFA World Cup own goals
FIFA World Cup
International Federation of Football History & Statistics


/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


['List of footballers with 500 or more goals',
 "List of men's footballers with 50 or more international goals",
 'List of footballers with more than 50 international goals',
 'List of footballers with 500 or more goals',
 "List of top international men's association football goal scorers by ...",
 'List of FIFA World Cup records and statistics',
 'List of footballers with 500 or more goals',
 'FIFA World Cup top goalscorers',
 'FIFA World Cup top goalscorers',
 'List of FIFA World Cup records and statistics']

In [21]:
docs,ids=retrieve_documents_eval(query,20)
res=[evidence_eval_df.loc[idx, 'title'] for idx in ids]
res


['List of footballers with 500 or more goals',
 "List of men's footballers with 50 or more international goals",
 'List of footballers with more than 50 international goals',
 'List of footballers with 500 or more goals',
 "List of top international men's association football goal scorers by ...",
 'List of FIFA World Cup records and statistics',
 'List of footballers with 500 or more goals',
 'Germany at the FIFA World Cup',
 'FIFA World Cup top goalscorers',
 'Football records and statistics in Spain',
 'FIFA World Cup top goalscorers',
 'List of FIFA World Cup records and statistics',
 'Football records and statistics in Spain',
 "List of top international men's association football goal scorers by ...",
 'List of FIFA World Cup records and statistics',
 'FIFA World Cup top goalscorers',
 'List of footballers with more than 50 international goals',
 'Football records and statistics in Spain',
 "List of top international men's association football goal scorers by ...",
 'Internationa

In [22]:
docs=[]
for title in titles:
    print(title)
    print(evidence_eval_df.loc[evidence_eval_df['title']==title,'text'].to_list())
    docs.extend(evidence_eval_df.loc[evidence_eval_df['title']==title,'text'])
print(docs)

List of men's footballers with 50 or more international goals
["Document: List of men's footballers with 50 or more international goals\n\n\n\nThe first player to score 50 international goals was Poul Nielsen of Denmark. He achieved this feat on his 36th cap against Sweden in the 1924–28 Nordic Football Championship on 14 June 1925. This is also the least number of matches played by any player to score 50 goals. He scored 52 goals in just 38 matches in his international career.[1][2][3][4]\nFerenc Puskás of Hungary was the second player and the second European after Nielsen to achieve the feat. His 50th goal came on 24 July 1952, when he scored a brace (two goals) in the semi-final match against Turkey at the 1952 Summer Olympics.[5] Puskás scored 84 goals in his international career.[6] He remained the highest international goalscorer for 47 years following his 84th goal in 1956 against Austria until Ali Daei of Iran broke the record in 2003 after scoring his 85th goal against Lebanon

In [23]:
def summarize(query, docs):
    prompt = """
    In a Retrieval Augmentation Generation system, documents close to the query vector are as follows:
    ---------------------
    {0}
    ---------------------
    Identify entities and contexts in a query, and use them to extract and summarize only relevant content from documents.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [24]:
def total_answer(query, docs):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    1. The query is an ambiguous question.
    2. Therefore, you must include the contents according to the various interpretations of the query in one answer by utilizing the given context.
    3. Each content according to the various interpretations of the query must be explained in one or two sentences.
    4. The total answer must be 5 sentences or less.
    Do not comment your answer and strictly follow this instructions.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [25]:
def various_answer(query, docs, first_ans=None):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    There may be multiple golden short answers in your answers, and they should be explained.
    Query: {1}{2}
    Answer:
    """.format('\n'.join(docs), query, f"Prior Answer: {first_ans}")
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [26]:
def answer(query, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [27]:
def retrieve_inside(query, titles, num=10):
    docs=[]
    text_docs=pd.DataFrame(columns=['text','title'])
    final_docs=[]
    for title in titles:
        text_docs=pd.concat([text_docs, evidence_eval_df.loc[evidence_eval_df['title']==title,['text','title']]],ignore_index=True)
        docs.extend(evidence_embeddings_eval[evidence_eval_df['title']==title])
        # print(title_retrieved_docs)
        # print()
    docs_tensor = torch.stack(docs)
    title_retrieved_docs,t=retrieve_documents2(query, docs_tensor, text_docs, num)
    final_docs.extend(title_retrieved_docs)
    return final_docs,t

In [28]:

set_seed(24)

docs=[]
first_ans=[]
for title in titles:
    # print(evidence_eval_df.loc[evidence_eval_df['title']==title,'text'].to_list())
    text_docs=evidence_eval_df.loc[evidence_eval_df['title']==title,'text'].to_list()
    docs_tensor=evidence_embeddings_eval[evidence_eval_df['title']==title]
    title_retrieved_docs,_=retrieve_documents3(query, docs_tensor, pd.DataFrame(text_docs, columns=['text']), 10)
    ans=answer(query,title_retrieved_docs)
    print(ans)
    first_ans.append(ans)
first_ans
final_ans=total_answer(query,first_ans)
final_ans

Ali Daei of Iran holds the record for the most international goals in world football with 109 goals.
Ali Daei of Iran has the highest number of goals in world football with 109 international goals.


KeyboardInterrupt: 

In [80]:
final_ans=total_answer(query,first_ans)
final_ans

'Based on the provided context, the player with the highest goals in world football is Josef Bican with 805 goals, but his total goals including youth-level statistics is not mentioned. However, Sebastián Abreu has a total of 837 goals, but this is not in world football. The player with the highest goals in international football is Ali Daei with 109 goals, and Abby Wambach has 184 international goals, making her the second-highest goalscorer. Miroslav Klose holds the record for the highest goals scored in World Cup history with 16 goals, but this is not the highest in world football. The highest goal scorer in world football is Josef Bican with 805 goals, but his youth-level statistics are not included.'

In [66]:
docs=[]
for title in titles:
    print(title)
    print(evidence_eval_df.loc[evidence_eval_df['title']==title,'text'].to_list())
    docs.extend(evidence_eval_df.loc[evidence_eval_df['title']==title,'text'])
print(docs)

List of men's footballers with 50 or more international goals
["Document: List of men's footballers with 50 or more international goals\n\n\n\nThe first player to score 50 international goals was Poul Nielsen of Denmark. He achieved this feat on his 36th cap against Sweden in the 1924–28 Nordic Football Championship on 14 June 1925. This is also the least number of matches played by any player to score 50 goals. He scored 52 goals in just 38 matches in his international career.[1][2][3][4]\nFerenc Puskás of Hungary was the second player and the second European after Nielsen to achieve the feat. His 50th goal came on 24 July 1952, when he scored a brace (two goals) in the semi-final match against Turkey at the 1952 Summer Olympics.[5] Puskás scored 84 goals in his international career.[6] He remained the highest international goalscorer for 47 years following his 84th goal in 1956 against Austria until Ali Daei of Iran broke the record in 2003 after scoring his 85th goal against Lebanon

In [23]:
# import torch
# from transformers import AutoModelForSequenceClassification, AutoTokenizer

# tokenizer_rerank = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')
# model_rerank = AutoModelForSequenceClassification.from_pretrained('BAAI/bge-reranker-v2-m3')
# model_rerank.eval()


In [24]:
# pairs = [['what is panda?', 'hi'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]
# with torch.no_grad():
#     inputs = tokenizer_rerank(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
#     scores = model(**inputs, return_dict=True).logits.view(-1, ).float()

# print(scores)

In [25]:
# from sentence_transformers import CrossEncoder

# v2m3_model = CrossEncoder("BAAI/bge-reranker-v2-m3", trust_remote_code=True)

# # Sample query and contexts
# query, contexts = ('Were Scott Derrickson and Ed Wood of the same nationality?',
#  [{'id': 0, 'text': 'Scott Derrickson Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer. He lives in Los Angeles, California. He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."'},
#  {'id': 1, 'text': 'Ed Wood Edward Davis Wood Jr. (October 10, 1924 - December 10, 1978) was an American filmmaker, actor, writer, producer, and director.'}
#  # 100 context in total ...
# ])
# context_texts = [t['text'] for t in contexts]

# def rerank_documents(model, query, contexts):
#  rets = model.rank(query, contexts, batch_size=64)
#  return rets


# rets=rerank_documents(v2m3_model, query, context_texts)

# Baseline

In [128]:
from tqdm import tqdm
from evaluation import evaluate

set_seed(24)

stop_iteration = 20
retrival_list=[]
scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs, doc_ids = retrieve_documents_eval(query,10)
    tmp=0
    for doc_id in doc_ids:    
        tmp+=match_sample_id(idx, doc_id)
    res=tmp/len(retrieved_docs)
    print("Retrival Match Rate:", res)
    dic=dict()
    dic['first_retrival']=res
    ans=total_answer(query,retrieved_docs)
    print('Final ans:', ans)
    scores=evaluate([ans], [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
    retrival_list.append(dic)
    retrival_df=pd.DataFrame(retrival_list)
    print(dic)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

retrival_df=pd.DataFrame(retrival_list)
retrival_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Retrival Match Rate: 0.5
Final ans: Ali Daei holds the record for the highest number of international goals with 109 goals for Iran, surpassing Ferenc Puskás' record of 84 goals. Cristiano Ronaldo has scored 1030 goals in his career, the highest among active players, with a total of 738 goals in his international career. Miroslav Klose holds the record for most World Cup goals with 16 goals, while Gerd Müller used to be the holder of that record from 1974 until it was broken by Ronaldo in 2006. Pelé scored 77 international goals, the highest among players from outside Europe, and Jürgen Klinsmann scored 11 goals in the World Cup, the highest among players from Germany. The top goalscorers in the World Cup history are Ali Daei, Cristiano Ronaldo, Ferenc Puskás, Kunishige Kamamoto, Godfrey Chitalu, Hussein Saeed, and Zainal Abidin, among others.
Ali Daei holds the record for the highest number of international goals with 109 goals for Iran, surpassing Ferenc Puskás' record of 84 goals. C

  5%|▌         | 1/20 [00:26<08:19, 26.31s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.6411338448524475, 'start': 0, 'end': 8, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.18989530205726624, 'start': 0, 'end': 8, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.05639030784368515, 'start': 0, 'end': 8, 'answer': 'Ali Daei'}
{'rougeLsum': 38.68312757201646, 'length': 136.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
{'first_retrival': 0.5}
Retrival Match Rate: 1.0
Final ans: The original artist of "Sound of Silence" is Simon & Garfunkel, an American music duo consisting of Paul Simon and Art Garfunkel. The song was written by Paul Simon, with the duo's original recording being a folk-rock acoustic version on

 10%|█         | 2/20 [00:43<06:18, 21.03s/it]

...

 95%|█████████▌| 19/20 [05:01<00:16, 16.14s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9669660329818726, 'start': 0, 'end': 12, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.7825772166252136, 'start': 99, 'end': 109, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.8997148871421814, 'start': 0, 'end': 12, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.6090750098228455, 'start': 99, 'end': 109, 'answer': 'Ethan Peck'}
{'rougeLsum': 31.16883116883117, 'length': 104.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
{'first_retrival': 1.0}
Retrival Match

100%|██████████| 20/20 [05:14<00:00, 15.71s/it]

follow question : Who is the 17th Chief Minister of MP?
short answer : ['Shivraj Singh Chauhan']
{'score': 0.004668977577239275, 'start': 48, 'end': 58, 'answer': 'Kamal Nath'}
follow question : Who is the 16th Chief Minister of MP?
short answer : ['Babulal Gaur']
{'score': 0.0011278680758550763, 'start': 48, 'end': 58, 'answer': 'Kamal Nath'}
follow question : Who is the 15th Chief Minister of MP?
short answer : ['Uma Bharti']
{'score': 0.00045598880387842655, 'start': 48, 'end': 58, 'answer': 'Kamal Nath'}
follow question : Who is the 17th chief minister of m. p?
short answer : ['Shivraj Singh Chauhan']
{'score': 0.0013139252550899982, 'start': 48, 'end': 58, 'answer': 'Kamal Nath'}
follow question : Who is the 16th chief minister of m. p?
short answer : ['Babulal Gaur', 'Babulal Gaur Yadav']
{'score': 0.00022961953072808683, 'start': 48, 'end': 58, 'answer': 'Kamal Nath'}
follow question : Who is the 15th chief minister of m. p?
short answer : ['Uma Bharti']
{'score': 9.390354534843

first_retrival    0.76
dtype: float64

In [130]:
# 기본 라그에 문서 10개 검색 함수는 기본 함수 answer
import pandas as pd
import math
sf = pd.read_csv('results/basicRAG_seed24-len20-newdata_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

20
rougeLsum      26.424077
length         31.200000
str_em         49.166667
Disambig-F1    44.894841
dtype: float64
34.44277482138087


In [132]:
# 기본 라그에 제목 검색 10개 후 그안에서 문서 10개 검색 함수는 기본함수 answer
import pandas as pd
import math
sf = pd.read_csv('results/basicRAG_seed24-len20-title_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

20
rougeLsum      30.492810
length         27.050000
str_em         48.750000
Disambig-F1    51.680556
dtype: float64
39.697422727900765


In [129]:
# 기본 라그에 문서 10개 검색 함수는 total_answer
import pandas as pd
import math
sf = pd.read_csv('results/basicRAG_seed24-totalanswer-doc10_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

20
rougeLsum      36.529148
length         93.200000
str_em         58.333333
Disambig-F1    49.708333
dtype: float64
42.61224056875744


In [131]:
# 기본 라그에 제목 검색 10개후 그안에서 문서 10개 검색 함수는 total_answer
import pandas as pd
import math
sf = pd.read_csv('results/basicRAG_seed24-totalanswer-title10_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

20
rougeLsum      36.918206
length         92.000000
str_em         56.666667
Disambig-F1    47.319444
dtype: float64
41.796519228254525


In [29]:
def answer_2020(query, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    1. If you can’t answer the question, say there is no relevant information.
    2. Answers to questions should be no longer than 2 sentences.
    3. When using time-related information to answer the question, only use information before February 1, 2020.
    Do not comment your answer and strictly follow this instructions.
    Query: {1}
    Answer:
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [30]:
from tqdm import tqdm
from evaluation import evaluate

set_seed(24)

stop_iteration = 20
retrival_list=[]
scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']   
    titles,title_ids=retrieve_titles(query, 10)
    # title_docs=find_docs(titles)
    docs=[]
    first_ans=[]
    for title in titles:
        # print(evidence_eval_df.loc[evidence_eval_df['title']==title,'text'].to_list())
        text_docs=evidence_eval_df.loc[evidence_eval_df['title']==title,'text'].to_list()
        docs_tensor=evidence_embeddings_eval[evidence_eval_df['title']==title]
        title_retrieved_docs,_=retrieve_documents3(query, docs_tensor, pd.DataFrame(text_docs, columns=['text']), 10)
        ans=answer_2020(query,title_retrieved_docs)
        print(ans)
        first_ans.append(ans)
    # final_docs, doc_titles=retrieve_inside(query, titles, 10)
    # print(final_docs, doc_titles)
    # # retrieved_docs, doc_ids = retrieve_documents2(query)
    # tmp=0
    # for doc_title in doc_titles:
    #     tmp+=match_sample_id2(idx, doc_title)
    # res=tmp/len(final_docs)
    # print("Retrival Match Rate:", res)
    # dic=dict()
    # dic['first_retrival']=res
    ans=total_answer(query,first_ans)
    print('Final ans:', ans)
    scores=evaluate([ans], [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
    # retrival_list.append(dic)
    # retrival_df=pd.DataFrame(retrival_list)
    # print(dic)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

# retrival_df=pd.DataFrame(retrival_list)
# retrival_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]

Ali Daei of Iran holds the record for the highest international goals in world football with 109 goals.
Romário has the highest goals in world football with 994 goals.
Ali Daei of Iran has the highest goals in world football with 109 international goals. He broke Ferenc Puskás' record of 84 goals in 2003.
Grzegorz Lato holds the record for the highest goals in world football with 20 goals.
Ali Daei has the highest goals in world football with 109 international goals.
Christine Sinclair has the highest goals in the list with 185 international goals.
There is no relevant information to answer this question as the provided context information is about UEFA Champions League top scorers, not world football.
There is no relevant information in the provided context about the highest goals in world football.
Miroslav Klose of Germany holds the record for most goals scored in the FIFA World Cup with 16 goals.
Miroslav Klose of Germany has the highest goals in world football with 16 goals.
Final

  5%|▌         | 1/20 [00:31<09:56, 31.37s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.10844346880912781, 'start': 0, 'end': 7, 'answer': 'Romário'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.00019743801385629922, 'start': 407, 'end': 425, 'answer': 'Christine Sinclair'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.9312098026275635, 'start': 407, 'end': 425, 'answer': 'Christine Sinclair'}
{'rougeLsum': 38.49765258215963, 'length': 109.0, 'str_em': 66.66666666666666, 'Disambig-F1': 33.33333333333333}
The original artist of "The Sound of Silence" is Simon & Garfunkel, a music duo consisting of Paul Simon and Art Garfunkel. The song was written by Paul Simon.
There is no relevant information about the original artist of the song "Sound of Silence" in the given cont

 10%|█         | 2/20 [00:59<08:54, 29.70s/it]

...

 95%|█████████▌| 19/20 [08:06<00:23, 23.60s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.24566921591758728, 'start': 169, 'end': 181, 'answer': 'Anthony Rapp'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.40464097261428833, 'start': 66, 'end': 76, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.21626503765583038, 'start': 66, 'end': 76, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.4185853898525238, 'start': 66, 'end': 76, 'answer': 'Ethan Peck'}
{'rougeLsum': 27.419354838709676, 'length': 83.0, 'str_em': 100.0, 'Disambig-F1': 50.0}
Kamal Nath of the Indian National Co

100%|██████████| 20/20 [08:33<00:00, 25.68s/it]

follow question : Who is the 17th Chief Minister of MP?
short answer : ['Shivraj Singh Chauhan']
{'score': 0.00030009355396032333, 'start': 310, 'end': 331, 'answer': 'Shivraj Singh Chouhan'}
follow question : Who is the 16th Chief Minister of MP?
short answer : ['Babulal Gaur']
{'score': 7.370475941570476e-05, 'start': 310, 'end': 331, 'answer': 'Shivraj Singh Chouhan'}
follow question : Who is the 15th Chief Minister of MP?
short answer : ['Uma Bharti']
{'score': 7.318148709600791e-05, 'start': 310, 'end': 331, 'answer': 'Shivraj Singh Chouhan'}
follow question : Who is the 17th chief minister of m. p?
short answer : ['Shivraj Singh Chauhan']
{'score': 2.953265266114613e-06, 'start': 310, 'end': 331, 'answer': 'Shivraj Singh Chouhan'}
follow question : Who is the 16th chief minister of m. p?
short answer : ['Babulal Gaur', 'Babulal Gaur Yadav']
{'score': 7.252684213199245e-07, 'start': 310, 'end': 331, 'answer': 'Shivraj Singh Chouhan'}
follow question : Who is the 15th chief ministe

rougeLsum       33.001145
length         243.250000
str_em          57.083333
Disambig-F1     47.782828
dtype: float64

In [31]:
scores_df.to_csv('./results/titleRAG_0215_results.csv', index=False)
import pandas as pd
import math
sf = pd.read_csv('results/titleRAG_0215_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

19
rougeLsum      34.561395
length         97.736842
str_em         56.578947
Disambig-F1    50.297714
dtype: float64
41.69363441522356


In [ ]:
import pandas as pd
import math
sf = pd.read_csv('results/basicRAG_seed24-len20-title_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

# Answer RAG

In [30]:
def hyde_query(query):
    prompt = """
    Create text that can answer the following question: Question: {0} Answer:
    """.format(query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 1024)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [ ]:
docs=[]
for _ in range(1):
    docs.append(hyde_query("Who has the highet goals in world football?"))
docs

["The highest goals in world football are held by Brazilian legend, Pelé, who scored an impressive 772 goals in 891 appearances throughout his career. However, it's essential to note that the accuracy of Pelé's goals may be disputed due to the lack of official records from his early years.The player with the second-highest goals in world football is Argentine legend, Lionel Messi, who has scored over 770 goals in around 912 appearances. Messi's impressive goal-scoring record has earned him numerous accolades, including seven Ballon d'Or awards.Other notable players who have achieved high goal-scoring records include:- Josef Bican (Austria): 805 goals in 529 appearances- Ferenc Puskás (Hungary): 746 goals in 629 appearances- Gerd Müller (Germany): 735 goals in 706 appearances- Roberto Baggio (Italy): 707 goals in 758 appearances- Alfredo Di Stéfano (Argentina/Spain): 694 goals in 659 appearancesThese players have all achieved incredible feats in the world of football, and their records 

In [34]:
def hyde_answer(ans):
    prompt = """
    Create a text that can ask questions in the following answers. Answer: {0} Question:
    """.format(ans)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 1024)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [35]:
res="Ali Daei holds the record for the highest number of international goals with 109 goals for Iran, surpassing Ferenc Puskás' record of 84 goals. Cristiano Ronaldo has scored 1030 goals in his career, the highest among active players, with a total of 738 goals in his international career. Miroslav Klose holds the record for most World Cup goals with 16 goals, while Gerd Müller used to be the holder of that record from 1974 until it was broken by Ronaldo in 2006. Pelé scored 77 international goals, the highest among players from outside Europe, and Jürgen Klinsmann scored 11 goals in the World Cup, the highest among players from Germany. The top goalscorers in the World Cup history are Ali Daei, Cristiano Ronaldo, Ferenc Puskás, Kunishige Kamamoto, Godfrey Chitalu, Hussein Saeed, and Zainal Abidin, among others."

In [36]:
ans=[]
for _ in range(1):
    ans.append(hyde_answer(res))
ans

["Here's a text with questions based on the given information:Football is a highly competitive sport, and records are often broken by new generations of players. Who holds the record for the highest number of international goals with 109 goals for Iran? Another notable record is held by Cristiano Ronaldo, who has scored a massive number of goals in his career. What is the total number of goals Cristiano Ronaldo has scored in his career? In the history of the World Cup, several players have stood out for their impressive goal-scoring skills. Who holds the record for most World Cup goals with 16 goals? Pelé is another football legend known for his incredible goal-scoring record. How many international goals did Pelé score? Some players have managed to break records that were previously held by other football legends. Who broke the record for most World Cup goals previously held by Gerd Müller? The top goalscorers in the World Cup history include many talented players from around the worl

In [42]:
def query_rewriting(query):
    prompt = """
    You are an AI assistant that improves user queries by rewriting ambiguous questions into clear and precise ones. When a question is unclear due to ambiguity, missing key elements, vague pronouns, overly broad scope, or lack of context, refine the question while preserving its original intent. Follow these steps:

    Identify ambiguity: Determine whether the question has multiple interpretations, missing elements, vague references, excessive generalization, or lacks context.
    Clarify missing details: If necessary, infer plausible missing information or add context to make the question specific.
    Eliminate vague pronouns: Replace words like "this," "that," or "it" with explicit references.
    Narrow overly broad questions: Specify the scope or include examples to make the question more actionable.
    Maintain intent: Ensure that the rewritten question conveys the same fundamental purpose as the original one.
    
    Example Cases and Rewritten Versions:
    Original: "How can I fix this?"
    Rewritten: "How can I fix the network connectivity issue on my laptop?"
    Original: "Why did he do that?"
    Rewritten: "Why did the project manager decide to change the deadline?"
    Original: "Is this theory applicable in real life?"
    Rewritten: "Can reinforcement learning be effectively applied to real-world robotics?"
    Original: "What is AI?"
    Rewritten: "What are the core principles of artificial intelligence in machine learning?"
    
    Now, rewrite the following ambiguous question into a clearer, more specific one:
    User's Question: {0}
    Rewritten Question:
    Do not comment your answer and strictly follow this instructions.
    """.format(query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 1024)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return re.sub('\n|<\|eot_id\|>', '', res)

In [43]:
ans=[]
for _ in range(5):
    ans.append(query_rewriting("who has the highet goals in world football?"))
ans

['Identify ambiguity: The question "who has the highest goals in world football" has multiple interpretations due to the ambiguity of "world football". It could refer to international football, domestic football leagues, or even a specific tournament.Clarify missing details: To make the question specific, we need to clarify the scope of "world football". Narrow overly broad questions: Let\'s assume the user is referring to international football. We\'ll also assume they\'re asking about the top scorer in the FIFA World Cup.Eliminate vague pronouns: No vague pronouns are present in this question.Maintain intent: The rewritten question conveys the same fundamental purpose as the original one: to find the top scorer in international football.Rewritten Question: "Who is the top scorer in the FIFA World Cup history?"',
 'Identify ambiguity: The question "who has the highest goals in world football" is ambiguous due to the lack of context and specificity. It\'s unclear whether the question r

In [37]:
from tqdm import tqdm
from evaluation import evaluate

set_seed(24)

stop_iteration = 20

scores_list=[]
retrival_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    
    retrieved_docs, doc_ids = retrieve_documents(query)
    tmp1=0
    for doc_id in doc_ids:    
        tmp1+=match_sample_id(idx, doc_id)
    first_retrival=tmp1/len(retrieved_docs)
    print("First Retrival Match Rate:", first_retrival)
    
    dic=dict()
    dic['first_retrival']=first_retrival
    
    first_ans=total_answer(query,retrieved_docs)
    print('First ans:', first_ans)
    
    # hyde_ans_list=hyde(query,first_ans)
    # print(hyde_ans_list)
    
    virtual_ans=hyde_answer(first_ans)
    print(virtual_ans)
    
    ans_docs, doc_ids=retrieve_documents(virtual_ans)
    
    tmp2=0
    for doc_id in doc_ids:    
        tmp2+=match_sample_id(idx, doc_id)
    
    second_retrival=tmp2/len(ans_docs)
    print("Second Retrival Match Rate:", second_retrival)
    
    dic['second_retrival']=second_retrival
    
    final_ans=total_answer(query,ans_docs)
    print('Second ans:', final_ans)
    scores=evaluate([final_ans], [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
    retrival_list.append(dic)
    retrival_df=pd.DataFrame(retrival_list)
    print(dic)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

retrival_df=pd.DataFrame(retrival_list)
retrival_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


First Retrival Match Rate: 0.5
First ans: Ali Daei holds the record for the highest number of international goals with 109 goals for Iran, surpassing Ferenc Puskás' record of 84 goals. Cristiano Ronaldo has scored 1030 goals in his career, the highest among active players, with a total of 738 goals in his international career. Miroslav Klose holds the record for most World Cup goals with 16 goals, while Gerd Müller used to be the holder of that record from 1974 until it was broken by Ronaldo in 2006. Pelé scored 77 international goals, the highest among players from outside Europe, and Jürgen Klinsmann scored 11 goals in the World Cup, the highest among players from Germany. The top goalscorers in the World Cup history are Ali Daei, Cristiano Ronaldo, Ferenc Puskás, Kunishige Kamamoto, Godfrey Chitalu, Hussein Saeed, and Zainal Abidin, among others.
Here's a text asking questions related to the information provided:In the world of football, several records have been set by talented pla

  5%|▌         | 1/20 [00:36<11:31, 36.38s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.5171696543693542, 'start': 0, 'end': 8, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.24958856403827667, 'start': 0, 'end': 8, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.0008715770090930164, 'start': 0, 'end': 8, 'answer': 'Ali Daei'}
{'rougeLsum': 31.57894736842105, 'length': 161.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
{'first_retrival': 0.5, 'second_retrival': 0.3}
First Retrival Match Rate: 1.0
First ans: The original artist of "Sound of Silence" is Simon & Garfunkel, an American music duo composed of Paul Simon and Art Garfunkel. The song was written by Paul Simon and was first released on their debut albu

 10%|█         | 2/20 [01:04<09:27, 31.51s/it]

...

follow question : As of 2017, how many times have the lakers won the finals?
short answer : ['16']
{'score': 0.511303186416626, 'start': 47, 'end': 49, 'answer': '16'}
follow question : As of 2016, how many times have the Lakers won the finals?
short answer : ['16']
{'score': 0.6982125043869019, 'start': 47, 'end': 49, 'answer': '16'}
follow question : As of 2015, how many times have the Lakers won the finals?
short answer : ['16']
{'score': 0.3959483206272125, 'start': 47, 'end': 49, 'answer': '16'}
{'rougeLsum': 33.93939393939394, 'length': 119.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
{'first_retrival': 0.4, 'second_retrival': 0.5}
First Retrival Match Rate: 0.7
First ans: The Indian National Congress is in power in six legislative assemblies: Punjab, Rajasthan, Chhattisgarh, Madhya Pradesh, Maharashtra (as part of the Maha Vikas Aghadi), and the union territory of Puducherry (in an alliance with DMK). The party is also in power in seven legislative assemblies as of July 2019: Punja

 50%|█████     | 10/20 [04:44<04:50, 29.08s/it]

follow question : How many states plus territories in india are under congress?
short answer : ['7']
{'score': 0.48130932450294495, 'start': 460, 'end': 463, 'answer': 'six'}
follow question : How many states alone in india are under congress?
short answer : ['5']
{'score': 0.11698012799024582, 'start': 460, 'end': 463, 'answer': 'six'}
{'rougeLsum': 20.000000000000004, 'length': 98.0, 'str_em': 0.0, 'Disambig-F1': 0.0}
{'first_retrival': 0.7, 'second_retrival': 0.6}
First Retrival Match Rate: 1.0
First ans: Fruma-Sarah is the late wife of Lazar Wolf, a wealthy butcher, who rises from the grave in Tevye's dream to warn him of severe retribution if Tzeitel marries Lazar. She is mentioned as a character in the musical, but not a major one, and is portrayed as a formidable figure who plays a key role in Tevye's decision-making process. Fruma-Sarah's presence in the story is a supernatural one, as she appears in Tevye's dream to convey her disapproval of Tzeitel's potential marriage to Laz

 50%|█████     | 10/20 [05:22<05:22, 32.24s/it]


KeyboardInterrupt: 

In [39]:
scores_df.to_csv('./results/0211answer_results.csv', index=False)
import pandas as pd
import math
sf = pd.read_csv('results/0211answer_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

19
rougeLsum      38.764659
length         86.526316
str_em         58.333333
Disambig-F1    45.701754
dtype: float64
42.09053249441862


In [ ]:
# from tqdm import tqdm
# from evaluation import evaluate

# set_seed(24)

# stop_iteration = 20

# scores_list=[]
# retrival_list=[]
# for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
#     if idx == stop_iteration: break
#     query = row['question']
    
#     retrieved_docs, doc_ids = retrieve_documents(query)
#     tmp1=0
#     for doc_id in doc_ids:    
#         tmp1+=match_sample_id(idx, doc_id)
#     first_retrival=tmp1/len(retrieved_docs)
#     print("First Retrival Match Rate:", first_retrival)
    
#     dic=dict()
#     dic['first_retrival']=first_retrival
    
#     first_ans=total_answer(query,retrieved_docs)
#     print('First ans:', first_ans)
    
#     # hyde_ans_list=hyde(query,first_ans)
#     # print(hyde_ans_list)
    
#     ans_docs, doc_ids=retrieve_documents(first_ans)
#     id_dic=dict()
#     tmp2=0
#     for doc_id in doc_ids:    
#         tmp2+=match_sample_id(idx, doc_id)
#         doc_sample_id=eval_df.loc[doc_id,'sample_id']
#         if doc_sample_id not in id_dic:
#             id_dic[doc_sample_id]=1
#         else:
#             id_dic[doc_sample_id]+=1
#     res=sorted(id_dic.items(), key= lambda item:-item[1])
#     print(res)
#     max_sample_id,max_value=res[0]
#     print("Max sample id:", max_sample_id)
#     # print(ans_docs)
    
#     if max_value>=5:
#         if max_sample_id==qa_df.loc[idx,'sample_id']:
#             second_retrival=1.0
#         else:
#             second_retrival=0.0
#         ans_docs=retrieve_documents2(query, max_sample_id)
#     else:
#         second_retrival=tmp2/len(ans_docs)
        
#     print("Second Retrival Match Rate:", second_retrival)
    
#     dic['second_retrival']=second_retrival
    
#     final_ans=total_answer(query,ans_docs)
#     print('Second ans:', final_ans)
#     scores=evaluate([final_ans], [row.to_dict()])
#     scores_list.append(scores)
#     scores_df=pd.DataFrame(scores_list)
#     print(scores)
#     retrival_list.append(dic)
#     retrival_df=pd.DataFrame(retrival_list)
#     print(dic)
        
# scores_df=pd.DataFrame(scores_list)
# scores_df.mean()

# retrival_df=pd.DataFrame(retrival_list)
# retrival_df.mean()

In [41]:
scores_df.mean()

rougeLsum      41.618101
length         90.600000
str_em         67.083333
Disambig-F1    51.569444
dtype: float64

In [19]:
scores_df.to_csv('./results/answer_rag_4_len60_results.csv', index=False)

In [3]:
# 기본 라그에서 total_answer함수써서 답변 생성 후 그 답변을 다시 검색해서 total_answer로 최종 답변 생성
import pandas as pd
import math
sf = pd.read_csv('results/answer_rag_4_len60_results.csv')
sf=sf[sf['length']<1000][:60]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

60
rougeLsum      38.451280
length         99.533333
str_em         64.583333
Disambig-F1    51.052730
dtype: float64
44.30623874568943


In [27]:
from tqdm import tqdm
from evaluation import evaluate

set_seed()

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    first_ans=answer(query,retrieved_docs)
    print('First ans:', first_ans[0])
    ans_docs=retrieve_documents(first_ans[0])
    final_ans=answer(first_ans[0],ans_docs)
    print('Second ans:', final_ans[0])
    scores=evaluate(final_ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


First ans: According to the provided context information, the player with the highest goals in world football is Ali Daei of Iran, with 109 goals in international matches.
Second ans: According to the provided context information, the player with the highest goals in world football is Ali Daei of Iran, with 109 goals in international matches.
According to the provided context information, the player with the highest goals in world football is Ali Daei of Iran, with 109 goals in international matches.
Who has the highest goals in world football?
["Who has the highest goals in men's world international football?", "Who has the highest goals all-time in men's football?", "Who has the highest goals in women's world international football?"]
[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'], ['Sinclair', 'Christine Sinclair']]


  5%|▌         | 1/20 [00:07<02:27,  7.74s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.8568584322929382, 'start': 102, 'end': 110, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.17466114461421967, 'start': 102, 'end': 110, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.05843370407819748, 'start': 102, 'end': 110, 'answer': 'Ali Daei'}
{'rougeLsum': 36.36363636363637, 'length': 26.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
First ans: Simon & Garfunkel
Second ans: Simon & Garfunkel was an American folk rock duo consisting of Paul Simon and Art Garfunkel. They were one of the most popular and influential musical acts of the 1960s, known for their harmonious vocals and introspective songwriting.Simon & Garf

 10%|█         | 2/20 [00:36<06:06, 20.37s/it]

...

 95%|█████████▌| 19/20 [03:06<00:09,  9.98s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9635234475135803, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 1.869488914962858e-05, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.8692940473556519, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.00026919867377728224, 'start': 68, 'end': 80, 'answer': 'Heath Ledger'}
{'rougeLsum': 42.10526315789474, 'length': 35.0, 'str_em': 50.0, 'Disambig-F1': 50.0}
First ans: No, Microsoft Live 

100%|██████████| 20/20 [03:14<00:00,  9.75s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.023067617788910866, 'start': 55, 'end': 63, 'answer': 'freeware'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.2880362868309021, 'start': 239, 'end': 255, 'answer': 'download and use'}
{'rougeLsum': 32.608695652173914, 'length': 56.0, 'str_em': 50.0, 'Disambig-F1': 50.0}


rougeLsum      33.400839
length         61.800000
str_em         48.333333
Disambig-F1    41.194444
dtype: float64

In [25]:
scores_df.to_csv('./results/answer_rag_2_results.csv', index=False)

In [26]:
import pandas as pd
import math
sf = pd.read_csv('results/answer_rag_2_results.csv')
sf=sf[sf['length']<1000][:100]
print(len(sf))
print(sf.mean())
r=sf.mean()['rougeLsum']
d=sf.mean()['Disambig-F1']
dr=math.sqrt(r*d)
print(dr)

16
rougeLsum      33.482502
length         97.562500
str_em         62.500000
Disambig-F1    42.708333
dtype: float64
37.815101059628596
